# LLM-105 Allegro Potential: GPU NVE Demonstration

**Validated in Google Colab on an NVIDIA T4 (13 September 2026).**

This notebook downloads a checksummed LAMMPS + `pair_nequip_allegro` executable and runs a short microcanonical (NVE) molecular-dynamics simulation with **fine-tuned model D (OMC25)**.

The 76-atom LLM-105 unit cell provides a short reproducibility test of the model and NVE workflow used in the HPC stability study. The repository also includes the unchanged paper-scale input for the 304-atom `2×1×2` supercell.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dantasqu/llm105-allegro-potentials/blob/main/notebooks/LLM105_Allegro_NVE_Colab.ipynb)

<details>
<summary><strong>Compatibility and provenance</strong></summary>

This model was validated with the software versions listed below. Changing these versions may cause compatibility errors.

| Component | Pinned value |
| --- | --- |
| PyTorch | 2.7.0 + CUDA 12.6, CXX11 ABI |
| LAMMPS | `e410a2816ac79be35d19e4b59cbfc9d7287308ba` (`stable_29Aug2024_update1`) |
| pair_nequip_allegro | `402b28390403aa92f34f14c2d5a9ff918acec598` |
| LAMMPS accelerator | Kokkos + CUDA, `allegro/kk` |
| Demonstration model | fine-tuned model D (OMC25) |
| Model SHA-256 | `d4103ddd2de524a33dcdef856af64021275e33fc0313072e1ce9c014b9905bc4` |

The 76-atom NVE smoke test completed successfully with GPU acceleration and LAMMPS exit code 0.

Official references: [Allegro LAMMPS integration](https://nequip.readthedocs.io/projects/allegro/en/latest/guide/lammps.html) and [pair_nequip_allegro](https://github.com/mir-group/pair_nequip_allegro).

</details>

## 1. Confirm a GPU runtime

Select **Runtime → Change runtime type → T4 GPU** before running the notebook.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU was detected. Enable a GPU runtime in Colab.")

subprocess.run(["nvidia-smi"], check=True)

## 2. Install the compatible runtime

In [ ]:
%%bash
set -euo pipefail
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openmpi-bin

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
     "torch==2.7.0", "--index-url", "https://download.pytorch.org/whl/cu126"],
    check=True,
)
print("Compatible CUDA PyTorch installed.")

In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CXX11 ABI:", torch._C._GLIBCXX_USE_CXX11_ABI)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

assert torch.__version__.startswith("2.7.0"), "Expected PyTorch 2.7.0."
assert torch.version.cuda == "12.6", "Expected the CUDA 12.6 PyTorch build."
assert torch._C._GLIBCXX_USE_CXX11_ABI is True, "The Kokkos build requires CXX11 ABI PyTorch."
assert torch.cuda.is_available(), "The CUDA-enabled PyTorch runtime cannot see a GPU."

## 3. Download LAMMPS with Allegro

Download and verify the prebuilt LAMMPS executable for an NVIDIA T4.

In [ ]:
from pathlib import Path
import hashlib
import shutil
import tarfile
import requests

OWNER = "dantasqu"
REPOSITORY = "llm105-allegro-potentials"
WORK_DIR = Path("/content/llm105_allegro_nve")
LMP = WORK_DIR / "lammps_kokkos/build/lmp"
CACHE_TAG = "colab-cache-v1"
CACHE_ASSET = "lammps-allegro-kokkos-t4-torch2.7-cu126-linux-x86_64.tar.gz"
EXPECTED_CACHE_SHA256 = "f39cf8364c9b325dbeb4f6a4a7c28f95a2487e802ad44d14b4a9fb41261db987"

gpu_cc = torch.cuda.get_device_capability(0)
if gpu_cc == (7, 5):
    cache_url = f"https://github.com/{OWNER}/{REPOSITORY}/releases/download/{CACHE_TAG}/{CACHE_ASSET}"
    asset_response = requests.get(cache_url, timeout=300)
    asset_response.raise_for_status()
    archive_path = Path("/content") / CACHE_ASSET
    archive_path.write_bytes(asset_response.content)
    digest = hashlib.sha256(asset_response.content).hexdigest()
    print("Cache SHA-256:", digest)
    assert digest == EXPECTED_CACHE_SHA256, "LAMMPS cache checksum mismatch."
    LMP.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as archive:
        member = next(item for item in archive.getmembers() if item.name.lstrip("./") == "lmp" and item.isfile())
        with archive.extractfile(member) as source, LMP.open("wb") as destination:
            shutil.copyfileobj(source, destination)
    LMP.chmod(0o755)
    print("CACHE_DOWNLOAD=PASS")
    print("Using verified prebuilt T4 executable:", LMP)
else:
    raise RuntimeError("This notebook requires an NVIDIA T4 GPU. Select T4 GPU under Runtime → Change runtime type.")

In [ ]:
import glob
import os
from pathlib import Path
import site
import subprocess
import torch

WORK_DIR = Path("/content/llm105_allegro_nve")
LMP = WORK_DIR / "lammps_kokkos/build/lmp"
torch_lib = Path(torch.__file__).resolve().parent / "lib"
nvidia_libs = [
    item
    for base in site.getsitepackages()
    for item in glob.glob(str(Path(base) / "nvidia/*/lib"))
]
runtime_env = os.environ.copy()
runtime_env["LD_LIBRARY_PATH"] = ":".join(
    [str(torch_lib), *nvidia_libs, runtime_env.get("LD_LIBRARY_PATH", "")]
)
runtime_env["OMP_NUM_THREADS"] = "1"

help_result = subprocess.run(
    [str(LMP), "-h"], capture_output=True, text=True, env=runtime_env, check=True
)
matching = [line for line in help_result.stdout.splitlines() if "allegro" in line.lower()]
print("\n".join(matching))
assert any("allegro/kk" in line.lower() for line in matching), "LAMMPS was built without allegro/kk."

source_directories = ((WORK_DIR / "lammps_kokkos", "LAMMPS"), (WORK_DIR / "pair_nequip_allegro", "pair_nequip_allegro"))
if all((directory / ".git").is_dir() for directory, _ in source_directories):
    for directory, label in source_directories:
        revision = subprocess.run(["git", "-C", str(directory), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
        print(f"{label} commit: {revision}")
else:
    print("Prebuilt cache provenance: LAMMPS e410a2816ac79be35d19e4b59cbfc9d7287308ba; pair_nequip_allegro 402b28390403aa92f34f14c2d5a9ff918acec598")

## 4. Download the model and input files

Download fine-tuned model D (OMC25), the NVE input, and the LLM-105 structure from the repository.

In [ ]:
from pathlib import Path
import requests

OWNER = "dantasqu"
REPOSITORY = "llm105-allegro-potentials"
REVISION = "main"
RUN_DIR = Path("/content/llm105_allegro_nve/run")
RUN_DIR.mkdir(parents=True, exist_ok=True)

MODEL_REPO_PATH = "models/fine_tuned/finetuned_model_d_omc25_llm105.nequip.pth"
INPUT_REPO_PATH = "examples/nve/in.nve"
STRUCTURE_REPO_PATH = "examples/nve/llm105_uc_std.data"
MODEL_PATH = RUN_DIR / "finetuned_model_d_omc25_llm105.nequip.pth"
INPUT_PATH = RUN_DIR / "in.nve"
STRUCTURE_PATH = RUN_DIR / "llm105_uc_std.data"

def download_repository_file(repository_path, destination):
    url = f"https://raw.githubusercontent.com/{OWNER}/{REPOSITORY}/{REVISION}/{repository_path}"
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    destination.write_bytes(response.content)
    print(f"Downloaded {repository_path} ({destination.stat().st_size:,} bytes)")

download_repository_file(MODEL_REPO_PATH, MODEL_PATH)
download_repository_file(INPUT_REPO_PATH, INPUT_PATH)
download_repository_file(STRUCTURE_REPO_PATH, STRUCTURE_PATH)

In [ ]:
import hashlib
import zipfile

EXPECTED_SHA256 = {
    MODEL_PATH: "d4103ddd2de524a33dcdef856af64021275e33fc0313072e1ce9c014b9905bc4",
    INPUT_PATH: "4205f6ef3032bd0384e532fb711f6c785bbddf3ed94c88a2e38635b90e1d7c2a",
    STRUCTURE_PATH: "24076d571bd2858578ed0682732777edba2267614626df4424cb8d75a74eea59",
}
for path, expected_digest in EXPECTED_SHA256.items():
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path.name} SHA-256: {digest}")
    assert digest == expected_digest, f"Checksum mismatch for {path.name}."

with zipfile.ZipFile(MODEL_PATH) as archive:
    names = archive.namelist()
    def read_extra(field):
        member = next(name for name in names if name.endswith(f"/extra/{field}"))
        return archive.read(member).decode().strip()
    metadata = {
        field: read_extra(field)
        for field in ["type_names", "num_types", "r_max", "model_dtype", "allow_tf32"]
    }

print("Embedded model metadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")
assert metadata["type_names"].split() == ["C", "H", "N", "O"]
assert metadata["num_types"] == "4"
assert metadata["r_max"] == "5.2"

## 5. Prepare the NVE simulation

Prepare a 76-atom NVE simulation at 300 K using fine-tuned model D (OMC25), with a short 0.1 fs gentle start followed by 1,000 production steps at 0.5 fs.

In [ ]:
KOKKOS_STRUCTURE_PATH = RUN_DIR / "llm105_uc_std_kokkos.data"
COLAB_INPUT_PATH = RUN_DIR / "in.colab.unit.nve"
GENTLE_STEPS = 20
PRODUCTION_STEPS = 1000

structure_lines = STRUCTURE_PATH.read_text().splitlines()
labels_start = structure_lines.index("Atom Type Labels")
masses_start = structure_lines.index("Masses")
clean_structure_lines = structure_lines[:labels_start] + structure_lines[masses_start:]
KOKKOS_STRUCTURE_PATH.write_text("\n".join(clean_structure_lines) + "\n")

adapted_lines = []
run_number = 0
changes = {"read_data": 0, "replicate": 0, "pair_coeff": 0, "minimize": 0, "run": 0}

for original_line in INPUT_PATH.read_text().splitlines():
    line = original_line
    stripped = line.strip()
    indentation = line[:len(line) - len(line.lstrip())]
    if stripped.startswith("read_data "):
        line = f"{indentation}read_data {KOKKOS_STRUCTURE_PATH}"
        changes["read_data"] += 1
    elif stripped.startswith("replicate "):
        changes["replicate"] += 1
        continue
    elif stripped.startswith("pair_coeff ") and ".nequip.pth" in stripped:
        tokens = stripped.split()
        model_index = next(i for i, token in enumerate(tokens) if token.endswith(".nequip.pth"))
        tokens[model_index] = str(MODEL_PATH)
        line = indentation + " ".join(tokens)
        changes["pair_coeff"] += 1
    elif stripped.startswith("minimize "):
        line = f"{indentation}minimize 1.0e-4 1.0e-6 5 20"
        changes["minimize"] += 1
    elif stripped.startswith("run "):
        run_number += 1
        steps = GENTLE_STEPS if run_number == 1 else PRODUCTION_STEPS
        line = f"{indentation}run {steps}"
        changes["run"] += 1
    adapted_lines.append(line)

assert changes == {"read_data": 1, "replicate": 1, "pair_coeff": 1, "minimize": 2, "run": 2}
COLAB_INPUT_PATH.write_text("\n".join(adapted_lines) + "\n")

print("NVE input prepared:", COLAB_INPUT_PATH)

## 6. Run the NVE simulation

Run the NVE simulation. The complete LAMMPS output is saved as `validation_success.log`.

In [ ]:
import time

VALIDATION_LOG = RUN_DIR / "validation_success.log"
peak_gpu_utilization = 0
peak_gpu_memory_mib = 0
start = time.perf_counter()

with VALIDATION_LOG.open("w") as log_handle:
    process = subprocess.Popen(
        [str(LMP), "-k", "on", "g", "1", "-sf", "kk",
         "-pk", "kokkos", "newton", "on", "neigh", "half",
         "-in", COLAB_INPUT_PATH.name],
        cwd=RUN_DIR,
        env=runtime_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    while process.poll() is None:
        sample = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip().split(",")
        peak_gpu_utilization = max(peak_gpu_utilization, int(sample[0].strip()))
        peak_gpu_memory_mib = max(peak_gpu_memory_mib, int(sample[1].strip()))
        time.sleep(0.5)

elapsed = time.perf_counter() - start
log_text = VALIDATION_LOG.read_text(errors="replace")
print("LAMMPS run summary")
print(f"  Exit code: {process.returncode}")
print(f"  Elapsed time: {elapsed:.1f} seconds")
print(f"  Peak GPU utilization: {peak_gpu_utilization}%")
print(f"  Peak GPU memory: {peak_gpu_memory_mib} MiB")

for prefix in ("Loop time of", "Performance:", "Total wall time:"):
    matches = [line.strip() for line in log_text.splitlines() if line.strip().startswith(prefix)]
    if matches:
        print(" ", matches[-1])

print("  Full log:", VALIDATION_LOG)

if process.returncode != 0:
    raise RuntimeError("LAMMPS did not complete successfully; inspect validation_success.log above.")
assert peak_gpu_memory_mib > 0, "No GPU memory use was observed during the run."

## 7. Results

The plots show temperature and total energy during the 0.5 ps production NVE segment.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def read_thermo_blocks(log_path):
    blocks = []
    header = None
    rows = []
    for raw_line in Path(log_path).read_text(errors="replace").splitlines():
        fields = raw_line.split()
        if fields and fields[0] == "Step" and "Temp" in fields:
            if header and rows:
                blocks.append(pd.DataFrame(rows, columns=header))
            header, rows = fields, []
            continue
        if header and len(fields) == len(header):
            try:
                rows.append([float(value) for value in fields])
            except ValueError:
                if rows:
                    blocks.append(pd.DataFrame(rows, columns=header))
                header, rows = None, []
    if header and rows:
        blocks.append(pd.DataFrame(rows, columns=header))
    return blocks

blocks = read_thermo_blocks(VALIDATION_LOG)
candidates = [block for block in blocks if len(block) >= 2 and "Temp" in block and "TotEng" in block]
if not candidates:
    raise ValueError("No multi-row thermo block containing Temp and TotEng was found.")
thermo = candidates[-1]

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(thermo["Step"], thermo["Temp"], marker="o", linewidth=1.2)
axes[0].set_ylabel("Temperature (K)")
axes[0].grid(alpha=0.25)
axes[1].plot(thermo["Step"], thermo["TotEng"], marker="o", linewidth=1.2)
axes[1].set_xlabel("LAMMPS step")
axes[1].set_ylabel("Total energy (eV)")
axes[1].grid(alpha=0.25)
fig.tight_layout()
plt.show()

n_atoms = 76
energy_drift_per_atom = (thermo["TotEng"].iloc[-1] - thermo["TotEng"].iloc[0]) / n_atoms
print(f"Initial production temperature: {thermo['Temp'].iloc[0]:.3f} K")
print(f"Final production temperature:   {thermo['Temp'].iloc[-1]:.3f} K")
print(f"Production total-energy drift: {energy_drift_per_atom:.6e} eV/atom")
assert thermo[["Temp", "PotEng", "KinEng", "TotEng"]].notna().all().all()

---

<details>
<summary><strong>Optional: build LAMMPS from source</strong></summary>

These commands reproduce the cached executable and are not executed by the notebook.

```bash
set -euo pipefail
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openmpi-bin libopenmpi-dev ninja-build
python -m pip install --quiet --upgrade torch==2.7.0 --index-url https://download.pytorch.org/whl/cu126

WORK_DIR=/content/llm105_allegro_nve
LAMMPS_DIR=${WORK_DIR}/lammps_kokkos
PAIR_DIR=${WORK_DIR}/pair_nequip_allegro
LAMMPS_COMMIT=e410a2816ac79be35d19e4b59cbfc9d7287308ba
PAIR_COMMIT=402b28390403aa92f34f14c2d5a9ff918acec598

mkdir -p "${WORK_DIR}"
if [[ ! -d "${LAMMPS_DIR}/.git" ]]; then
  git init "${LAMMPS_DIR}"
  git -C "${LAMMPS_DIR}" remote add origin https://github.com/lammps/lammps.git
  git -C "${LAMMPS_DIR}" fetch --depth 1 origin "${LAMMPS_COMMIT}"
  git -C "${LAMMPS_DIR}" checkout --detach FETCH_HEAD
fi

if [[ ! -d "${PAIR_DIR}/.git" ]]; then
  git init "${PAIR_DIR}"
  git -C "${PAIR_DIR}" remote add origin https://github.com/mir-group/pair_nequip_allegro.git
  git -C "${PAIR_DIR}" fetch --depth 1 origin "${PAIR_COMMIT}"
  git -C "${PAIR_DIR}" checkout --detach FETCH_HEAD
fi

test "$(git -C "${LAMMPS_DIR}" rev-parse HEAD)" = "${LAMMPS_COMMIT}"
test "$(git -C "${PAIR_DIR}" rev-parse HEAD)" = "${PAIR_COMMIT}"

if ! grep -q 'find_package(Torch REQUIRED)' "${LAMMPS_DIR}/cmake/CMakeLists.txt"; then
  (cd "${PAIR_DIR}" && ./patch_lammps.sh "${LAMMPS_DIR}")
else
  echo "LAMMPS source is already patched for PyTorch."
fi

GPU_CC=$(python -c 'import torch; print("".join(map(str, torch.cuda.get_device_capability(0))))')
case "${GPU_CC}" in
  70) KOKKOS_ARCH=VOLTA70 ;;
  75) KOKKOS_ARCH=TURING75 ;;
  80) KOKKOS_ARCH=AMPERE80 ;;
  86) KOKKOS_ARCH=AMPERE86 ;;
  89) KOKKOS_ARCH=ADA89 ;;
  90) KOKKOS_ARCH=HOPPER90 ;;
  *) echo "Unsupported GPU compute capability ${GPU_CC}" >&2; exit 1 ;;
esac
echo "Kokkos GPU architecture: ${KOKKOS_ARCH}"

TORCH_CMAKE_PREFIX=$(python -c 'import torch; print(torch.utils.cmake_prefix_path)')
NVCC_WRAPPER=${LAMMPS_DIR}/lib/kokkos/bin/nvcc_wrapper

cmake -S "${LAMMPS_DIR}/cmake" -B "${LAMMPS_DIR}/build" -G Ninja \
  -D CMAKE_BUILD_TYPE=Release \
  -D CMAKE_CXX_STANDARD=17 \
  -D CMAKE_CXX_COMPILER="${NVCC_WRAPPER}" \
  -D CMAKE_PREFIX_PATH="${TORCH_CMAKE_PREFIX}" \
  -D BUILD_MPI=ON \
  -D BUILD_OMP=ON \
  -D PKG_MOLECULE=ON \
  -D PKG_KOKKOS=ON \
  -D Kokkos_ENABLE_CUDA=ON \
  -D Kokkos_ENABLE_SERIAL=ON \
  -D Kokkos_ARCH_${KOKKOS_ARCH}=ON \
  -D NEQUIP_AOT_COMPILE=OFF \
  -D MKL_INCLUDE_DIR=/tmp

cmake --build "${LAMMPS_DIR}/build" --target lmp --parallel 2
test -x "${LAMMPS_DIR}/build/lmp"
echo "LAMMPS executable: ${LAMMPS_DIR}/build/lmp"
```
</details>